In [ ]:
from typing import Dict

def parse_molecular_formula(formula: str)-> Dict[str, int]:
    atom_dict=dict()
    for i in range(len(formula)):
        if i<len(formula) and formula[i].isalpha() and formula[i].isupper():  #C, K (one letter symbol)
            atomic_symbol=formula[i]
            atom_count=1

            if i+1<len(formula) and formula[i+1].islower():    #Al, Na (two letter symbol)
                atomic_symbol=formula[i]+formula[i+1]
                digits=""
                for j in range(i+2, len(formula)):             #Al3 (two letter symbol with digit afterwards)
                    if formula[j].isdigit():
                     digits=(digits+formula[j])        
                    else:
                     break                                     #AlC, AlCl, NaCl (two letter symbol then no digit)
                if digits != "":    
                    atom_count=int(digits) 

            elif i+1<len(formula) and formula[i+1].isupper():   #CH, OH  (one letter symbol next to each other with no digit in between)
                atom_count= 1

            elif i+1<len(formula) and formula[i+1].isdigit():   #C2, O2, K4 (one letter symbol with digit next to it)
                digits=""
                for j in range(i+1, len(formula)):
                    if formula[j].isdigit():                    #C234 (one letter symbol with more than one digit)
                     digits=(digits+formula[j])         
                    else: break 
                atom_count=int(digits)

            elif i==len(formula)-1 and formula[i].isalpha():     #C.....K (last letter with no digit afterwards, count as =1)
                atom_count=1
                       
            atom_dict[atomic_symbol]=atom_dict.get(atomic_symbol, 0) + atom_count
    return atom_dict 


# Lecture 2: Computational Thinking with Python

## Exercise 2: Chemical Equation Balancer Checker

**Objective:** Write a program that checks if a chemical equation is balanced.

**Problem:** Create a function called `is_balanced()` that takes a chemical equation string and returns True if balanced, False if not.

**Your Task:**
- Split the equation at the arrow symbol "→" to get reactants and products
- For each side, parse the molecules and their coefficients
- Count total atoms of each element on both sides
- Compare the totals

Example Equation Format: "2H2 + O2 → 2H2O"

**Hints:**

- Use equation.split("→") to separate reactants from products
- Use side.split(" + ") to separate individual molecules
- You'll need to handle coefficients (numbers in front of molecules)
- Reuse your count_atoms() function from Exercise 1!
- For molecules with coefficients like "2H2O", the coefficient applies to all atoms in that molecule

**Steps to Think Through:**

- How do you extract the coefficient from "2H2O"? (It's everything before the first letter)
- How do you multiply the atom counts by the coefficient?
- How do you combine counts from multiple molecules on the same side?


In [ ]:
#IDEATION

#to split the reaction into lhs and rhs
#to split the lhs and rhs side into separate molecules through "+"
# strip any extra spaces if given in the reaction
# for lhs side : extract all the atoms and and counts
# extract the coefficient and multiply it with coefficient to get final formula
#similarly for RHS and then check if both sides match


In [ ]:
def is_balanced(chem_eq: str) -> bool:
    
    lhs,rhs=chem_eq.split("→")                 # splitting and stripping
    lhs_side=lhs.strip().split("+")
    rhs_side=rhs.strip().split("+")

#for LHS
    lhs_atoms = {} 
    lhs_parts = [term.strip() for term in lhs_side] 
    for i in range(len(lhs_side)):
        if lhs_parts[i][0].isdigit():
           counts=""
           for j in range(0,len(lhs_parts[i])):
            if lhs_parts[i][j].isdigit() :
             counts= counts+lhs_parts[i][j] 
            else: break
           coeff=int(counts)
           formula=(lhs_parts[i][len(counts):]) 
        else: 
           coeff=1 
           formula=(lhs_parts[i])

        atom_dict = parse_molecular_formula(formula)     

        for symbol in atom_dict:
          total = atom_dict[symbol] * coeff            
          lhs_atoms[symbol] = lhs_atoms.get(symbol, 0) + total

#for RHS
    rhs_atoms = {} 
    rhs_parts = [term.strip() for term in rhs_side]
    for i in range(len(rhs_side)):
        if rhs_parts[i][0].isdigit():
           counts=""
           for j in range(0,len(rhs_parts[i])):
            if rhs_parts[i][j].isdigit() :
             counts= counts+rhs_parts[i][j] 
            else: break
           coeff=int(counts)
           formula=(rhs_parts[i][len(counts):]) 
        else: 
           coeff=1 
           formula=(rhs_parts[i]) 
                                         

        atom_dict = parse_molecular_formula(formula)     

        for symbol in atom_dict:
          total = atom_dict[symbol] * coeff           
          rhs_atoms[symbol] = rhs_atoms.get(symbol, 0) + total


    #print(f"{lhs_atoms}: {rhs_atoms} {'✓, balanced equation' if lhs_atoms == rhs_atoms else '✗, unbalanced equation'}")
    return lhs_atoms==rhs_atoms

       





In [ ]:
chem_eq="2H2 + O2 → 2H2O"
is_balanced(chem_eq)

In [ ]:
# chem_eq="2H2 + O2 → 2H2O"
# lhs,rhs=chem_eq.split("→")
# lhs=lhs.strip()
# lhs_parts=lhs.split("+")
# rhs=rhs.strip()
# rhs_parts=rhs.split("+")

# print(lhs)
# print(rhs)
# print(lhs_parts)
# print(rhs_parts)
# print(len(lhs_parts))
# print(len(rhs_parts))

### Test Cases


In [ ]:
# If the outcome is incorrect, the keyword assert will cause an error

print(is_balanced("2H2 + O2 → 2H2O"))
assert is_balanced("2H2 + O2 → 2H2O")

print(is_balanced("H2 + O2 → H2O"))
assert not is_balanced("H2 + O2 → H2O")

print(is_balanced("CH4 + 2O2 → CO2 + 2H2O"))
assert is_balanced("CH4 + 2O2 → CO2 + 2H2O")

print(is_balanced("N2 + H2 → NH3"))
assert not is_balanced("N2 + H2 → NH3")

## Exercise 3: Reaction Pathway Analyzer

**Objective:** Analyze multi-step reaction sequences to find synthesis pathways.

**Problem:** Write a program that determines what compounds can be synthesized from starting materials through a series of reactions, and finds a pathway to create a target compound.

**Your Task:**
Create a function called `find_synthesis_path()` that takes:

- A list of starting compounds
- A list of reaction equations
- A target compound to synthesize

The function should return either:

- A list of reaction steps that lead to the target compound, OR
- A message saying the target cannot be synthesized

**Hints:**

- Start by creating a set of "available" compounds (your starting materials)
- For each reaction, check if you have all the reactants needed
- If you can perform a reaction, add the products to your available compounds
- Keep track of which reactions you've used in what order
- Continue until you either find the target or can't make any new compounds

#### Computational Thinking:

**Decomposition:** Break this into smaller problems

- Parse reaction equations
- Track available compounds
- Check if reactions can be performed
- Record the synthesis pathway


**Pattern Recognition:** What patterns do you see?

- Each reaction has the same format: "reactants → products"
- You need to repeatedly check if new reactions become possible


**Algorithm Design:** What's your step-by-step approach?

- Initialize available compounds
- Loop: try each reaction, see if possible, update available compounds
- Stop when target is found or no progress is made


**Abstraction:** What functions might be helpful?

- A function to parse a single reaction equation
- A function to check if a reaction is possible with current compounds
- A function to update available compounds after a reaction



In [ ]:
#IDEATION
#extract the target equation that contains target by exyracting RHS from all equations
#and marking the one where it matches
#extract a list excludingg those targetreactions
#split the target reaction in  lhs and rhs and usingthe obive two functions xtract the finak formula
#multiplied wth coefiicents
#IDEA IS: (target reaction LHS)-->subset--->(excluded reaction lhs)+(excluded reaction rhs)+
#                                            (starting materials)
#For the one step reaction, getting thereaction containing target molecule in RHS
#for multistep, getting all the reactions above the target making sure that reaction is possible
# or not possible reactions: (target reactio LHS)--->is not subset--->(excluded reaction lhs)+(excluded reaction rhs)+
#                                            (starting materials)

In [ ]:
#ROUGH CODE
# starting_materials = ["C", "H2", "O2", "H2O"]
# reactions = [
#     "C + O2 → CO2",
#     "2H2 + O2 → 2H2O",
#     "CO2 + H2O → H2CO3",
#     "C + 2H2 → CH4",
#     "CH4 + 2O2 → CO2 + 2H2O"
# ]
# target = "CO2"
# target_reaction=[]

# for x in range(len(reactions)):
#     chem_eq=reactions[x]

#     lhs,rhs=chem_eq.split("→")
#     lhs_side=lhs.strip().split("+")
#     rhs_side=rhs.strip().split("+")

# # TO CHECK IF THE TARGET IS PRESENT OR NOT
# #for RHS
#     rhs_atoms = {} 
#     rhs_parts = [term.strip() for term in rhs_side]
#     for i in range(len(rhs_side)):
#         if rhs_parts[i][0].isdigit():
#            counts=""
#            for j in range(0,len(rhs_parts[i])):
#             if rhs_parts[i][j].isdigit() :
#              counts= counts+rhs_parts[i][j] 
#             else: break
#            coeff=int(counts)
#            formula=(rhs_parts[i][len(counts):]) 
#         else: 
#            coeff=1 
#            formula=(rhs_parts[i])

        
#         if formula==target:
#           target_reaction.append(reactions[x])
# print(target_reaction)



# # TO EXTRACT LHS FROM TARGET REACTION 
# t_lhs_elements=[]  
# for t in target_reaction: 
#    t_lhs,t_rhs= t.split("→")
#    t_lhs_side=t_lhs.strip().split("+")

#    t_lhs_parts = [term.strip() for term in t_lhs_side]
#    for i in range(len(t_lhs_side)):
#         if t_lhs_parts[i][0].isdigit():
#            counts=""
#            for j in range(0,len(t_lhs_parts[i])):
#             if t_lhs_parts[i][j].isdigit() :
#              counts= counts+t_lhs_parts[i][j] 
#             else: break
#            coeff=int(counts)
#            formula=(t_lhs_parts[i][len(counts):])
#         else: 
#            coeff=1 
#            formula=(t_lhs_parts[i])
#         t_lhs_elements.append(formula)  

# print(t_lhs_elements)         


# # TO EXTRACT ATOMS FROM EXCLUDED REACTION 
# reactions_excl_target_eq = [r for r in reactions if r not in target_reaction]
  
# for ex in reactions_excl_target_eq: 
#    ex_lhs,ex_rhs= ex.split("→")
#    ex_lhs_side=ex_lhs.strip().split("+")
#    ex_rhs_side=ex_rhs.strip().split("+")
#    ex_list=ex_lhs_side+ex_rhs_side+starting_materials


# excluded_elements=[]  
# ex_list_parts = [term.strip() for term in ex_list]
# for i in range(len(ex_list)):
#         if ex_list_parts[i][0].isdigit():
#            counts=""
#            for j in range(0,len(ex_list_parts[i])):
#             if ex_list_parts[i][j].isdigit() :
#              counts= counts+ex_list_parts[i][j] 
#             else: break
#            coeff=int(counts)
#            formula=(ex_list_parts[i][len(counts):])
#         else: 
#            coeff=1 
#            formula=(ex_list_parts[i])
#         excluded_elements.append(formula)  

# print(excluded_elements) 


# is_subset= set(t_lhs_elements).issubset(excluded_elements)
# print(is_subset)



          


                             

#### Fianl Function

In [ ]:
from typing import List

def find_synthesis_path(starting_materials: List[str], reactions: List[str], target: str) -> List[str] | str:

 target_reaction=[]

 for x in range(len(reactions)):
    chem_eq=reactions[x]

    lhs,rhs=chem_eq.split("→")
    lhs_side=lhs.strip().split("+")
    rhs_side=rhs.strip().split("+")

# TO CHECK IF THE TARGET IS PRESENT OR NOT
#for RHS
    rhs_atoms = {} 
    rhs_parts = [term.strip() for term in rhs_side]
    for i in range(len(rhs_side)):
        if rhs_parts[i][0].isdigit():
           counts=""
           for j in range(0,len(rhs_parts[i])):
            if rhs_parts[i][j].isdigit() :
             counts= counts+rhs_parts[i][j] 
            else: break
           coeff=int(counts)
           formula=(rhs_parts[i][len(counts):]) 
        else: 
           coeff=1 
           formula=(rhs_parts[i])

        
        if formula==target:
          target_reaction.append(reactions[x])
 print("target reaction: ",target_reaction)



# TO EXTRACT LHS FROM TARGET REACTION 
 t_lhs_elements=[]  
 for t in target_reaction: 
   t_lhs,t_rhs= t.split("→")
   t_lhs_side=t_lhs.strip().split("+")

   t_lhs_parts = [term.strip() for term in t_lhs_side]
   for i in range(len(t_lhs_side)):
        if t_lhs_parts[i][0].isdigit():
           counts=""
           for j in range(0,len(t_lhs_parts[i])):
            if t_lhs_parts[i][j].isdigit() :
             counts= counts+t_lhs_parts[i][j] 
            else: break
           coeff=int(counts)
           formula=(t_lhs_parts[i][len(counts):])
        else: 
           coeff=1 
           formula=(t_lhs_parts[i])
        t_lhs_elements.append(formula)  

 #print(t_lhs_elements)         


# TO EXTRACT ATOMS FROM EXCLUDED REACTION 
 reactions_excl_target_eq = [r for r in reactions if r not in target_reaction]
 ex_list=list(starting_materials) 
 for ex in reactions_excl_target_eq: 
   ex_lhs,ex_rhs= ex.split("→")
   ex_lhs_side=ex_lhs.strip().split("+")
   ex_rhs_side=ex_rhs.strip().split("+")
   ex_list=ex_list+ex_lhs_side+ex_rhs_side


 excluded_elements=[]  
 ex_list_parts = [term.strip() for term in ex_list]
 for i in range(len(ex_list)):
        if ex_list_parts[i][0].isdigit():
           counts=""
           for j in range(0,len(ex_list_parts[i])):
            if ex_list_parts[i][j].isdigit() :
             counts= counts+ex_list_parts[i][j] 
            else: break
           coeff=int(counts)
           formula=(ex_list_parts[i][len(counts):])
        else: 
           coeff=1 
           formula=(ex_list_parts[i])
        excluded_elements.append(formula)  

 #print(excluded_elements) 

  
 is_subset= set(t_lhs_elements).issubset(excluded_elements)
 if is_subset==True:
    print(f"{target} can be synthesized")
    print ("reactions of complete the synthesis are: ",list(dict.fromkeys(reactions[:x] + target_reaction)))
 else: print(f"{target} cannot be synthesized")
 return is_subset
                             

    

### Test Cases

In [ ]:
# Basic Multi-Step Synthesis

starting_materials = ["H2", "O2", "N2"]
reactions = [
    "2H2 + O2 → 2H2O",
    "N2 + 3H2 → 2NH3", 
    "NH3 + H2O → NH4OH"
]
target = "NH4OH"

result = find_synthesis_path(starting_materials, reactions, target)
print(result)
# Expected: Should find a path using H2 + N2 → NH3, then NH3 + H2O → NH4OH

In [ ]:
# Direct Synthesis (One Step)

starting_materials = ["Na", "Cl2"]
reactions = [
    "2Na + Cl2 → 2NaCl",
    "NaCl + H2O → NaOH + HCl"
]
target = "NaCl"

result = find_synthesis_path(starting_materials, reactions, target)
print(result)
# Expected: Should find direct synthesis in one step

In [ ]:
# Impossible Synthesis

starting_materials = ["H2", "O2"]
reactions = [
    "2H2 + O2 → 2H2O",
    "N2 + 3H2 → 2NH3"  # But we don't have N2!
]
target = "NH3"

result = find_synthesis_path(starting_materials, reactions, target)
print(result)
# Expected: Should return message that NH3 cannot be synthesized

In [ ]:
# Multiple Pathway Options
starting_materials = ["C", "H2", "O2", "H2O"]
reactions = [
    "C + O2 → CO2",
    "2H2 + O2 → 2H2O",
    "CO2 + H2O → H2CO3",
    "C + 2H2 → CH4",
    "CH4 + 2O2 → CO2 + 2H2O"
]
target = "CO2"

result = find_synthesis_path(starting_materials, reactions, target)
print(result)
# Expected: Should find one of multiple possible paths to CO2

In [ ]:
# Long Chain Synthesis
starting_materials = ["Fe", "O2", "C"]
reactions = [
    "4Fe + 3O2 → 2Fe2O3",
    "2C + O2 → 2CO",
    "Fe2O3 + 3CO → 2Fe + 3CO2",
    "CO2 + C → 2CO"
]
target = "CO2"

result = find_synthesis_path(starting_materials, reactions, target)
print(result)
# Expected: Should find the multi-step path involving iron oxide formation and reduction